# Step 3: Attention

## 3.1 Ne Değişiyor?

Önceki model, sonraki tokenı yalnızca mevcut tokena göre
tahmin ediyordu. Yeni model, konum bilgisini ve önceki
tokenlardan gelen bilgileri de kullanacak.

Bu bölümde konum embeddingleri, RMSNorm, tek başlı attention,
residual bağlantılar, çıktı katmanı ve KV cache eklenecek.
Veri kümesi, tokenizer, Value sınıfı ve SGD yöntemi korunacak.

In [1]:
import random
import math
with open("../data/raw_names.txt", encoding="utf-8") as f:
    docs = [
        line.strip()
        .replace("İ", "i")
        .replace("I", "ı")
        .lower()
        .replace(" ", "")
        for line in f
        if line.strip()
    ]

random.shuffle(docs)

uchars = sorted(set(''.join(docs)))
BOS = len(uchars)
vocab_size = len(uchars) + 1

print("İsim sayısı:", len(docs))
print("Sözlük boyutu:", vocab_size)

İsim sayısı: 1100
Sözlük boyutu: 30


## 3.2 Konum Embeddingleri

Token embeddingi, hangi tokenın işlendiğini temsil eder.
Konum embeddingi ise tokenın dizideki konumunu temsil eder.

Bu iki vektör eleman eleman toplanır. Böylece aynı token,
farklı konumlarda farklı giriş vektörleriyle temsil edilir.
Her iki embedding tablosu da eğitim sırasında öğrenilir.

In [2]:



class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(
            self.data * other.data,
            (self, other),
            (other.data, self.data)
        )

    def __pow__(self, other):
        return Value(
            self.data ** other,
            (self,),
            (other * self.data ** (other - 1),)
        )

    def log(self):
        return Value(
            math.log(self.data),
            (self,),
            (1 / self.data,)
        )

    def exp(self):
        return Value(
            math.exp(self.data),
            (self,),
            (math.exp(self.data),)
        )

    def relu(self):
        return Value(
            max(0, self.data),
            (self,),
            (float(self.data > 0),)
        )

    def __neg__(self):
        return self * -1

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other ** -1

    def __rtruediv__(self, other):
        return other * self ** -1

    # 2.5 Backward
    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad

In [3]:

n_embd = 16

block_size = 16

matrix = lambda nout, nin: [
    [Value(random.gauss(0, 0.08)) for _ in range(nin)]
    for _ in range(nout)
]

state_dict = {
    'wte': matrix(vocab_size, n_embd),
    'wpe': matrix(block_size, n_embd),
}

In [4]:
token_id = uchars.index("e")
pos_id = 1

tok_emb = state_dict['wte'][token_id]
pos_emb = state_dict['wpe'][pos_id]

x = [t + p for t, p in zip(tok_emb, pos_emb)]

print("Token embeddingi:", [v.data for v in tok_emb[:3]])
print("Konum embeddingi:", [v.data for v in pos_emb[:3]])
print("Birleşik vektör:", [v.data for v in x[:3]])
print("Vektör uzunluğu:", len(x))

Token embeddingi: [-0.0819113263683266, -0.0006639014990065998, 0.05269825877124451]
Konum embeddingi: [-0.13840293946677817, -0.004860294993178743, 0.045980142526008995]
Birleşik vektör: [-0.22031426583510477, -0.005524196492185342, 0.0986784012972535]
Vektör uzunluğu: 16
